In [77]:
import os
from dotenv import load_dotenv

API Keys

In [78]:
load_dotenv()

True

Embedding model from hugging face

In [79]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

LLM from GROQ

In [80]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama3-70b-8192")

VectorStore

In [81]:
from langchain_chroma import Chroma
vectorstore = Chroma(persist_directory="chroma_legal_db", embedding_function=embedding_model)

Retriver

In [82]:
retriever = vectorstore.as_retriever()

History aware Retriver

In [83]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate

contextualize_system_prompt = ( 
    "Using chat history and the latest user question, just reformulate question if needed and otherwise return it as is"
)

contextualize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",contextualize_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    llm,retriever,contextualize_prompt
)



Rag with History

In [84]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

system_prompt = (
    "You are an intelligent chatbot that can answer to business and coperate law in sri lanka, use the following context to answer the question. If you dont know the answer just say that you dont know."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}"),
    ]
)

Chains

In [85]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

qa_chain = create_stuff_documents_chain(llm,prompt)
rag_chain = create_retrieval_chain(history_aware_retriever,qa_chain)

Managing History

In [86]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str)-> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

In [87]:
response = conversational_rag_chain.invoke(
    {"input":"what are the laws that i need to consider when creating a Shareholder Agreement ?"},
    config={"configurable":{"session_id":"101"}},
)
print(response["answer"])

When creating a Shareholder Agreement in Sri Lanka, you should consider the following laws and provisions:

1. **Companies Act, No. 07 of 2007**:
	* Section 144: Written resolutions by shareholders, which can be as valid as if passed at a meeting.
	* Section 182: Power of the court to make orders in respect of the conduct of a company's affairs.
	* Section 226: Application to the court by shareholders in respect of the company's affairs.
2. **Shareholder rights and liabilities**:
	* Section 87: Liability of shareholders is limited to any liability expressly provided for in the articles of the company or under the Companies Act.
	* Section 88: Liability of shareholders in respect of calls or other liabilities attaching to shares.
3. **Alteration of shareholder rights**:
	* Section 144(6): Where a company fails to comply with the requirements of sending a copy of the resolution to every shareholder who did not sign the resolution.
4. **Shareholder protection**:
	* Section 182: The court'

In [89]:
response = conversational_rag_chain.invoke(
    {"input":"Is there terms for share holder agreement ?"},
    config={"configurable":{"session_id":"101"}},
)
print(response["answer"])

Yes, a Shareholder Agreement typically includes several key terms and provisions to govern the relationship between the shareholders and the company. Here are some common terms and provisions:

1. **Definitions**: Define key terms used in the agreement, such as "Share," "Shareholder," "Company," and "Board."
2. **Share Capital**: Specify the authorized share capital, number of shares, and par value of each share.
3. **Shareholding**: List the shareholders and their respective shareholdings.
4. **Voting Rights**: Outline the voting rights of each shareholder, including the number of votes per share and any restrictions on voting.
5. **Board Composition**: Determine the composition of the Board of Directors, including the number of directors, their roles, and how they will be appointed or removed.
6. **Management and Control**: Specify the management structure and decision-making process, including the roles and responsibilities of the Board and management team.
7. **Restrictions on Tran